# 이전 예제를 Sub classing 네트워크 설계방식 적용

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 구조 변경(차원)
print(x_train.shape)    # (60000,, 28, 28)
x_train = x_train.reshape((-1, 28, 28, 1)).astype('float32') / 255.0
x_test = x_test.reshape((-1, 28, 28, 1)).astype('float32') / 255.0
print(x_train.shape)    # (60000, 28, 28, 1)

(60000, 28, 28)
(60000, 28, 28, 1)


In [3]:
# model
# 사용자 정의 클래스(모델, 레이어, 함수: 손실, 활성화)를 모델 저장시 자동으로 직렬화 시스템에 등록해 주는 역할
from re import X


@tf.keras.utils.register_keras_serializable(package='custom')
class MyMnistCnn(tf.keras.Model):
    def __init__(self, **kwargs):     # self를 모르면 파이썬을 모르는거다
        super().__init__(**kwargs)
        # Conv block1
        self.conv1 = tf.keras.layers.Conv2D(16, (3,3), padding='same', activation='relu')
        self.pool1 = tf.keras.layers.MaxPool2D((2, 2))
        # Conv block2
        self.conv2 = tf.keras.layers.Conv2D(32, (3,3), padding='same', activation='relu')
        self.pool2 = tf.keras.layers.MaxPool2D((2, 2))
        # Conv block3
        self.conv3 = tf.keras.layers.Conv2D(64, (3,3), padding='same', activation='relu')
        self.pool3 = tf.keras.layers.MaxPool2D((2, 2))

        self.flat = tf.keras.layers.Flatten()

        self.d1 = tf.keras.layers.Dense(64, activation='relu')
        self.do1 = tf.keras.layers.Dropout(0.3)
        self.d2 = tf.keras.layers.Dense(32, activation='relu')
        self.do2 = tf.keras.layers.Dropout(0.2)
        self.out = tf.keras.layers.Dense(10, activation='softmax')
    
    def call(self, inputs, training=False):
        x = self.conv1(inputs)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.conv3(x)
        x = self.pool3(x)
        x = self.flat(x)
        x = self.d1(x)
        x = self.do1(x, training=True)
        x = self.d2(x)
        x = self.do2(x, training=True)
        return self.out(x)

model = MyMnistCnn()
model.build(input_shape=(None, 28, 28, 1))
model.summary()



Model: "my_mnist_cnn"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             multiple                  160       
                                                                 
 max_pooling2d (MaxPooling2D  multiple                 0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           multiple                  4640      
                                                                 
 max_pooling2d_1 (MaxPooling  multiple                 0         
 2D)                                                             
                                                                 
 conv2d_2 (Conv2D)           multiple                  18496     
                                                                 
 max_pooling2d_2 (MaxPooling  multiple                

In [4]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
es = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)

history = model.fit(
    x_train, y_train, epochs=100, batch_size=128, validation_split=0.1, callbacks=[es], verbose=2
)

# 모델 평가
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'train_loss:{train_loss:.4f}, train_acc:{train_acc:.4f}')
print(f'test_loss:{train_loss:.4f}, test_acc:{train_acc:.4f}')

Epoch 1/100
422/422 - 6s - loss: 0.5699 - accuracy: 0.8151 - val_loss: 0.1819 - val_accuracy: 0.9493 - 6s/epoch - 15ms/step
Epoch 2/100
422/422 - 1s - loss: 0.1634 - accuracy: 0.9544 - val_loss: 0.1123 - val_accuracy: 0.9703 - 1s/epoch - 3ms/step
Epoch 3/100
422/422 - 1s - loss: 0.1146 - accuracy: 0.9686 - val_loss: 0.0893 - val_accuracy: 0.9752 - 1s/epoch - 3ms/step
Epoch 4/100
422/422 - 1s - loss: 0.0944 - accuracy: 0.9742 - val_loss: 0.0738 - val_accuracy: 0.9817 - 1s/epoch - 3ms/step
Epoch 5/100
422/422 - 1s - loss: 0.0771 - accuracy: 0.9785 - val_loss: 0.0776 - val_accuracy: 0.9778 - 1s/epoch - 3ms/step
Epoch 6/100
422/422 - 1s - loss: 0.0691 - accuracy: 0.9808 - val_loss: 0.0628 - val_accuracy: 0.9842 - 1s/epoch - 3ms/step
Epoch 7/100
422/422 - 1s - loss: 0.0592 - accuracy: 0.9836 - val_loss: 0.0685 - val_accuracy: 0.9832 - 1s/epoch - 3ms/step
Epoch 8/100
422/422 - 1s - loss: 0.0533 - accuracy: 0.9851 - val_loss: 0.0593 - val_accuracy: 0.9853 - 1s/epoch - 3ms/step
Epoch 9/100
422